# Regresión Multinomial Bayesiana — Compradores del Movistar Arena

Este cuaderno implementa una **regresión logística multinomial bayesiana** con PyMC para clasificar compradores de boletas del concierto de Miguel Amezquita en el Movistar Arena dentro de tres perfiles:

| Clase | Código | Descripción |
|-------|--------|-------------|
| `Planner` | 0 | Compra con semanas de anticipación, comportamiento planificado |
| `In-Between` | 1 | Ni planificador ni compulsivo, grupo indeciso |
| `Last-Minute` | 2 | Compra a último momento, impulsado por urgencia |

**Objetivo:** No solo clasificar, sino entender *qué características* del cliente y su compra se asocian con cada tipo de comportamiento.

**Modelo matemático (del taller):**
$$P(Y_i = c \mid X_i) = \frac{\exp(\alpha_c + X_i^\prime \beta_c)}{\sum_{j=1}^{3} \exp(\alpha_j + X_i^\prime \beta_j)}, \quad c \in \{\text{Planner, In-Between, Last-Minute}\}$$

Donde:
- $\alpha_c$ es el intercepto de la clase $c$
- $\beta_c$ es el vector de coeficientes para la clase $c$
- La función **softmax** garantiza que las tres probabilidades sumen 1

**Facultad de Ciencia de Datos — Universidad Externado de Colombia**

## 0. Instalación de dependencias (Google Colab)

Si estás en Google Colab, descomenta la siguiente celda para instalar las librerías necesarias.

In [ ]:
# Descomenta si estás en Google Colab
# !pip install pymc arviz openpyxl scikit-learn -q

## 1. Importaciones y configuración

Importamos todas las librerías necesarias al inicio del cuaderno, siguiendo el estilo del profesor.

In [ ]:
# numpy: operaciones numéricas vectorizadas (arrays, matrices)
import numpy as np

# pandas: lectura y manipulación de datos tabulares (DataFrames)
import pandas as pd

# matplotlib: librería base de visualización en Python
import matplotlib.pyplot as plt

# seaborn: visualización estadística de alto nivel, construida sobre matplotlib
import seaborn as sns

# pymc: librería principal para modelado probabilístico bayesiano con MCMC
import pymc as pm

# arviz: librería para análisis e interpretación de trazas bayesianas (diagnósticos y plots)
import arviz as az

# StandardScaler: estandariza variables (media=0, desv=1) para mejorar la convergencia MCMC
from sklearn.preprocessing import StandardScaler, LabelEncoder

# confusion_matrix, classification_report: métricas de evaluación del clasificador
from sklearn.metrics import confusion_matrix, classification_report

# train_test_split: dividir datos en entrenamiento y prueba
from sklearn.model_selection import train_test_split

# warnings: suprimir advertencias menores que no afectan los resultados
import warnings
warnings.filterwarnings('ignore')

# os: manejo de directorios del sistema (para crear carpetas de figuras)
import os

# Configuración visual: estilo limpio con grilla blanca, resolución adecuada
sns.set_theme(style='whitegrid')        # fondo blanco con grilla gris suave
plt.rcParams['figure.dpi'] = 100       # resolución de figuras en pantalla

# Carpeta donde se guardarán todas las figuras generadas
IMG_DIR = '../img/resultados_movistar'
os.makedirs(IMG_DIR, exist_ok=True)    # crea la carpeta si no existe; no falla si ya existe

# Verificar versiones instaladas
print(f'PyMC versión:  {pm.__version__}')
print(f'ArviZ versión: {az.__version__}')
print(f'NumPy versión: {np.__version__}')

## 2. Carga y EDA

### 2.1 Lectura del dataset

El archivo `movistar.xlsx` contiene 1000 registros de compradores del concierto. Cada fila es un cliente único con sus características de compra.

In [ ]:
# Ruta al archivo de datos — ajusta si ejecutas desde Google Colab
DATA_PATH = '../data/movistar.xlsx'

# pd.read_excel: lee el archivo xlsx y lo convierte en un DataFrame de pandas
# Cada columna del Excel se convierte en una Serie con su tipo de dato automáticamente
df = pd.read_excel(DATA_PATH)

# Información básica del dataset
print(f'Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'\nTipos de variables:')
print(df.dtypes)
print(f'\nValores nulos por columna:')
print(df.isnull().sum())

# Mostrar los primeros 5 registros
df.head()

### 2.2 Estadísticas descriptivas

Calculamos estadísticas descriptivas **separadas por tipo de cliente** para entender las diferencias entre grupos antes de modelar.

In [ ]:
# Estadísticas generales de todas las variables numéricas
print('=== Estadísticas globales ===')
display(df.describe().round(2))

# Agrupamos por Customer_Type y calculamos la media de cada variable numérica
# Esto nos permite ver QUÉ DIFERENCIA hay entre Planner, In-Between y Last-Minute
print('\n=== Media por tipo de cliente ===')
display(
    df.groupby('Customer_Type')[[
        'Age', 'Num_Tickets_Purchased', 'Ticket_Price',
        'Concession_Purchases', 'Days_Before_Concierto'
    ]].mean().round(2)
)

# Distribución de la variable objetivo (conteo de clientes por clase)
print('\n=== Distribución de Customer_Type ===')
print(df['Customer_Type'].value_counts())
print(f'\nPorcentajes:')
print((df['Customer_Type'].value_counts(normalize=True) * 100).round(1))

### 2.3 Distribución de la variable objetivo

In [ ]:
# Conteo de clientes por tipo: verificamos si el dataset está balanceado
# Un dataset desbalanceado puede sesgar el modelo hacia la clase mayoritaria
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Countplot: barras con el conteo de cada categoría
conteo = df['Customer_Type'].value_counts()
axes[0].bar(conteo.index, conteo.values, color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='white')
axes[0].set_title('Conteo por tipo de cliente')
axes[0].set_xlabel('Customer_Type')
axes[0].set_ylabel('Número de clientes')
for i, (tipo, val) in enumerate(conteo.items()):
    axes[0].text(i, val + 5, str(val), ha='center', fontweight='bold')

# Pie chart: proporción de cada clase
axes[1].pie(
    conteo.values, labels=conteo.index,
    autopct='%1.1f%%', colors=['#2196F3', '#FF9800', '#4CAF50'],
    startangle=90
)
axes[1].set_title('Proporción por tipo de cliente')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/01_distribucion_clases.png', bbox_inches='tight')
plt.show()
print('Guardado: 01_distribucion_clases.png')

### 2.4 Boxplots de predictores por tipo de cliente

Los boxplots permiten ver visualmente qué variables discriminan mejor entre los tres tipos de comprador. La variable más importante será aquella cuyos boxplots **no se solapan** entre clases.

In [ ]:
# Variables numéricas continuas a comparar por clase
vars_continuas = [
    'Days_Before_Concierto',  # ¿cuántos días antes compraron?
    'Ticket_Price',           # ¿cuánto pagaron por boleta?
    'Age',                    # ¿qué edad tienen?
    'Concession_Purchases',   # ¿cuánto gastaron en concesiones?
    'Num_Tickets_Purchased'   # ¿cuántas boletas compraron?
]

# Orden de las clases para que aparezcan siempre en el mismo orden
orden_clases = ['Planner', 'In-Between', 'Last-Minute']
paleta = {'Planner': '#2196F3', 'In-Between': '#FF9800', 'Last-Minute': '#4CAF50'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()  # aplanar la matriz de ejes para iterar fácilmente

for i, var in enumerate(vars_continuas):
    # sns.boxplot: muestra mediana, cuartiles Q1/Q3 y outliers por grupo
    sns.boxplot(
        x='Customer_Type', y=var, data=df,
        order=orden_clases, palette=paleta, ax=axes[i]
    )
    axes[i].set_title(f'{var} por tipo de cliente')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=15)

# Ocultar el sexto panel (hay 5 variables, 6 posiciones en grilla 2×3)
axes[5].set_visible(False)

plt.suptitle('Distribución de variables por tipo de comprador', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/02_boxplots_por_clase.png', bbox_inches='tight')
plt.show()
print('Guardado: 02_boxplots_por_clase.png')

### 2.5 Matriz de correlación

In [ ]:
# Codificación temporal de variables categóricas solo para calcular correlación
df_corr = df.copy()
df_corr['fan_num'] = (df_corr['Fan_Mailing_List'] == 'Yes').astype(int)
df_corr['seat_num'] = (df_corr['Seat_Location'] == 'Lower').astype(int)

# Variables numéricas para la matriz de correlación
cols_num = ['Age', 'Num_Tickets_Purchased', 'Ticket_Price',
            'Concession_Purchases', 'Days_Before_Concierto', 'fan_num', 'seat_num']
corr = df_corr[cols_num].corr()

# Heatmap: colores cálidos = correlación positiva, fríos = negativa
plt.figure(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))  # ocultar triángulo superior (simétrico)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, square=True,
    linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Matriz de Correlación de Pearson — Predictores', fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/03_heatmap_correlacion.png', bbox_inches='tight')
plt.show()
print('Guardado: 03_heatmap_correlacion.png')

### 2.6 Violin plots de Days_Before_Concierto

`Days_Before_Concierto` es con alta probabilidad el predictor más discriminante. Los **Planner** compran con muchos días de anticipación, los **Last-Minute** con 0-2 días. Visualizamos la distribución completa (no solo la mediana) con violin plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Violin plot: muestra la densidad de probabilidad completa, no solo boxplot
# La anchura en cada punto indica cuántos datos hay ahí
sns.violinplot(
    x='Customer_Type', y='Days_Before_Concierto', data=df,
    order=orden_clases, palette=paleta, ax=axes[0], inner='box'
)
axes[0].set_title('Days_Before_Concierto por tipo de cliente')
axes[0].set_xlabel('Tipo de cliente')
axes[0].set_ylabel('Días de anticipación')

# Fan_Mailing_List: ¿los fans planifican más?
# Countplot agrupado: fan vs no fan dentro de cada tipo de cliente
fan_counts = df.groupby(['Customer_Type', 'Fan_Mailing_List']).size().reset_index(name='count')
sns.barplot(
    data=fan_counts, x='Customer_Type', y='count',
    hue='Fan_Mailing_List', order=orden_clases,
    palette={'Yes': '#9C27B0', 'No': '#BDBDBD'}, ax=axes[1]
)
axes[1].set_title('Fan Mailing List por tipo de cliente')
axes[1].set_xlabel('Tipo de cliente')
axes[1].set_ylabel('Número de clientes')
axes[1].legend(title='Fan')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/04_violin_dias_fan.png', bbox_inches='tight')
plt.show()
print('Guardado: 04_violin_dias_fan.png')

## 3. Preprocesamiento

Preparamos los datos para PyMC. Seguimos el mismo principio que el profesor en el cuaderno de regresión lineal:
- Variables continuas → estandarizar con `StandardScaler` para mejorar la exploración MCMC
- Variables binarias → codificar como 0/1
- Variable objetivo → codificar como enteros 0, 1, 2

In [ ]:
# ---- CODIFICACIÓN DE VARIABLES CATEGÓRICAS ----

# Fan_Mailing_List: 'Yes' → 1, 'No' → 0
# Esto convierte la variable de texto a numérica binaria que PyMC puede procesar
df['fan_int'] = (df['Fan_Mailing_List'] == 'Yes').astype(int)

# Seat_Location: 'Lower' → 1, 'Upper' → 0
# Lower es la sillería premium (más cara), codificamos como 1 para que
# el coeficiente represente el efecto de estar en zona Lower
df['seat_lower'] = (df['Seat_Location'] == 'Lower').astype(int)

# ---- CODIFICACIÓN DE LA VARIABLE OBJETIVO ----

# Customer_Type tiene 3 categorías. PyMC necesita enteros 0, 1, 2
# Asignamos: Planner=0, In-Between=1, Last-Minute=2
# Esta asignación es arbitraria; el modelo no asume ningún orden entre clases
mapa_clases = {'Planner': 0, 'In-Between': 1, 'Last-Minute': 2}
df['y_encoded'] = df['Customer_Type'].map(mapa_clases)

# Verificamos que la codificación es correcta
print('Verificación de codificación de la variable objetivo:')
print(df[['Customer_Type', 'y_encoded']].value_counts().sort_index())

# ---- ESTANDARIZACIÓN DE VARIABLES CONTINUAS ----

# Variables continuas que se estandarizarán
# Estandarizar significa restar la media y dividir por la desviación estándar:
#   z = (x - media) / std
# Esto hace que todas las variables tengan media≈0 y desv≈1,
# lo que permite que el sampler NUTS explore el espacio posterior de forma eficiente
# (si las variables tienen escalas muy distintas, el gradiente es difícil de calibrar)
predictores_continuos = [
    'Age',                    # edad del comprador
    'Days_Before_Concierto',  # días de anticipación (predictor más discriminante)
    'Ticket_Price',           # precio pagado por boleta
    'Concession_Purchases',   # gasto en concesiones
    'Num_Tickets_Purchased'   # cantidad de boletas
]

# StandardScaler: aprende media y std del conjunto de entrenamiento y transforma
scaler = StandardScaler()
X_continuas_z = scaler.fit_transform(df[predictores_continuos])
X_continuas_df = pd.DataFrame(
    X_continuas_z,
    columns=[f'{c}_z' for c in predictores_continuos]  # renombramos con sufijo _z
)

# Variables binarias (ya están en escala 0/1, no necesitan estandarización)
predictores_binarios = ['fan_int', 'seat_lower']

# Concatenar todas las variables predictoras en una sola matriz X
X_df = pd.concat(
    [X_continuas_df, df[predictores_binarios].reset_index(drop=True)],
    axis=1   # axis=1 → pegar columnas (no filas)
)

# Convertir a array numpy para PyMC (requiere arrays, no DataFrames)
X_data = X_df.values.astype('float64')  # float64 para precisión numérica
y_data = df['y_encoded'].values          # array de enteros: 0, 1 o 2

# Dimensiones del problema
n_obs       = X_data.shape[0]   # número de observaciones (clientes)
n_pred      = X_data.shape[1]   # número de predictores
n_clases    = 3                  # número de categorías de Customer_Type

print(f'\nDimensiones del problema:')
print(f'  Observaciones (n):  {n_obs}')
print(f'  Predictores (p):    {n_pred}  → {X_df.columns.tolist()}')
print(f'  Clases (C):         {n_clases} → {list(mapa_clases.keys())}')
print(f'  Parámetros totales: {n_clases} interceptos + {n_pred}×{n_clases} coeficientes = {n_clases + n_pred*n_clases}')

print(f'\nEstadísticas de X estandarizado:')
display(X_df.describe().round(3))

### 3.1 División train/test estratificada

Separamos el 20% de los datos como conjunto de prueba para evaluar el modelo más adelante. `stratify=y_data` garantiza que la proporción de cada clase sea igual en train y test.

In [ ]:
# train_test_split: divide aleatoriamente los datos en dos conjuntos
# test_size=0.2 → 20% para evaluación, 80% para entrenamiento del modelo bayesiano
# stratify=y_data → mantiene la proporción de clases en ambos conjuntos
# random_state=42 → reproducibilidad (siempre la misma división)
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data,
    test_size=0.2,
    stratify=y_data,
    random_state=42
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} observaciones')
print(f'Conjunto de prueba:        {X_test.shape[0]} observaciones')
print(f'\nDistribución de clases en train:')
clases_nombre = {v: k for k, v in mapa_clases.items()}
for c in [0, 1, 2]:
    n = (y_train == c).sum()
    print(f'  {clases_nombre[c]:12s}: {n} ({n/len(y_train)*100:.1f}%)')
print(f'\nDistribución de clases en test:')
for c in [0, 1, 2]:
    n = (y_test == c).sum()
    print(f'  {clases_nombre[c]:12s}: {n} ({n/len(y_test)*100:.1f}%)')

## 4. Modelo Bayesiano con PyMC

### Especificación matemática completa

Siguiendo el estilo del profesor (cuadernos `RegresionLineal.ipynb` y `Modelo_Multivariado_1.ipynb`), definimos el modelo usando `pm.Normal` para los priors y `pm.Categorical` como verosimilitud.

**Priors:**
$$\alpha_c \sim N(0, 5^2), \quad c \in \{0, 1, 2\}$$
$$\beta_{p,c} \sim N(0, 5^2), \quad p = 1, \ldots, 7; \quad c \in \{0, 1, 2\}$$

**Modelo lineal (logits):**
$$\eta_{i,c} = \alpha_c + \sum_{p=1}^{7} X_{i,p} \cdot \beta_{p,c}$$

**Función softmax (probabilidades):**
$$p_{i,c} = \text{softmax}(\eta_i)_c = \frac{\exp(\eta_{i,c})}{\sum_{j=0}^{2} \exp(\eta_{i,j})}$$

**Verosimilitud:**
$$Y_i \sim \text{Categorical}(p_{i,0}, p_{i,1}, p_{i,2})$$

### ¿Por qué priors N(0, 5²)?
Como el profesor usa distribuciones no informativas cuando no hay datos históricos, usamos $N(0, 5^2)$ — un prior amplio que no domina los datos pero sí estabiliza la exploración MCMC.

In [ ]:
# Construimos el modelo con PyMC, tal cual el estilo del profesor
# El bloque 'with pm.Model() as nombre_modelo:' crea el contexto del modelo.
# Todo lo definido dentro de este bloque pertenece al modelo.

with pm.Model() as modelo_multinomial:

    # ------------------------------------------------------------------
    # PRIORS DE LOS INTERCEPTOS
    # ------------------------------------------------------------------
    # alpha tiene shape=(n_clases,) → un intercepto por cada clase (3 en total)
    # alpha[0] → intercepto para Planner
    # alpha[1] → intercepto para In-Between
    # alpha[2] → intercepto para Last-Minute
    # mu=0: centramos en cero porque no tenemos información previa sobre qué clase es más probable
    # sigma=5: prior amplio (débilmente informativo), permite que los datos muevan el intercepto libremente
    alpha = pm.Normal(
        'alpha',          # nombre interno del parámetro en el trace
        mu=0,             # media del prior: sin sesgo inicial hacia ninguna clase
        sigma=5,          # desviación del prior: amplia para no dominar los datos
        shape=n_clases    # un escalar por cada clase → vector de longitud 3
    )

    # ------------------------------------------------------------------
    # PRIORS DE LOS COEFICIENTES
    # ------------------------------------------------------------------
    # beta tiene shape=(n_pred, n_clases) → una matriz de coeficientes
    # beta[j, c] = efecto del predictor j sobre la clase c
    # Por ejemplo: beta[1, 0] = efecto de Days_Before_Concierto_z sobre ser Planner
    # mu=0: sin dirección a priori para ningún predictor
    # sigma=5: igual que alpha, débilmente informativo
    # Como los predictores están estandarizados, beta=1 significa que
    # una desviación estándar del predictor mueve el logit en 1 unidad
    beta = pm.Normal(
        'beta',                     # nombre en el trace
        mu=0,                       # sin sesgo a priori
        sigma=5,                    # prior amplio
        shape=(n_pred, n_clases)    # matriz: predictores × clases
    )

    # ------------------------------------------------------------------
    # MODELO LINEAL — CÁLCULO DE LOGITS
    # ------------------------------------------------------------------
    # X_train tiene forma (n_obs_train, n_pred)
    # beta tiene forma (n_pred, n_clases)
    # pm.math.dot(X_train, beta) realiza la multiplicación matricial:
    #   resultado tiene forma (n_obs_train, n_clases)
    #   cada fila i contiene los logits η_{i,c} para las 3 clases
    # Al sumar alpha (forma n_clases), numpy broadcast suma el intercepto de cada clase
    # El resultado mu[i, c] = α_c + X_i · β_c (producto punto del predictor con los coef de clase c)
    mu = pm.math.dot(X_train, beta) + alpha
    # mu tiene forma (n_obs_train, 3): un logit por observación y por clase

    # ------------------------------------------------------------------
    # FUNCIÓN SOFTMAX — CONVERSIÓN A PROBABILIDADES
    # ------------------------------------------------------------------
    # La función softmax toma los logits η y los convierte en probabilidades:
    #   p_{i,c} = exp(η_{i,c}) / Σ_j exp(η_{i,j})
    # Propiedades:
    #   1. Cada p_{i,c} ∈ (0, 1): valores entre 0 y 1
    #   2. Σ_c p_{i,c} = 1: las tres probabilidades suman exactamente 1
    # axis=-1: aplica el softmax a lo largo del último eje (las clases),
    #   es decir, normaliza por fila (por observación)
    p = pm.math.softmax(mu, axis=-1)
    # p tiene forma (n_obs_train, 3): probabilidades de pertenecer a cada clase

    # ------------------------------------------------------------------
    # VEROSIMILITUD — DISTRIBUCIÓN CATEGORICAL
    # ------------------------------------------------------------------
    # pm.Categorical: distribución para variables categóricas discretas
    # Es el equivalente multinomial para una sola observación
    # p=p: las probabilidades calculadas por softmax para cada observación
    # observed=y_train: los valores reales de Customer_Type (0, 1 o 2)
    # Al pasar 'observed', PyMC sabe que debe calcular la verosimilitud
    # P(Y_i = y_i | p_i) para cada observación i
    y_obs = pm.Categorical(
        'y_obs',          # nombre interno
        p=p,              # probabilidades softmax para cada observación
        observed=y_train  # datos observados: etiquetas reales 0, 1, 2
    )

# Confirmamos que el modelo está definido correctamente
print('Modelo especificado correctamente.')
print(f'Variables libres: {[rv.name for rv in modelo_multinomial.free_RVs]}')
print(f'Variables observadas: {[rv.name for rv in modelo_multinomial.observed_RVs]}')

### 4.1 Muestreo MCMC — NUTS

Usamos el algoritmo **NUTS** (No-U-Turn Sampler), que es el sampler por defecto de PyMC y el que el profesor utiliza en todos sus ejemplos. NUTS es una versión adaptativa de Hamiltonian Monte Carlo (HMC) que:

- Usa el **gradiente** de la log-posterior para proponer movimientos eficientes
- Evita el problema de "U-turn" (cuando las propuestas regresan al punto de partida)
- Converge mucho más rápido que Metropolis-Hastings para modelos multidimensionales

**Parámetros:**
- `draws=50000`: número de muestras a guardar **por cadena** → 200,000 muestras totales
- `tune=5000`: pasos de adaptación donde NUTS aprende el tamaño de paso óptimo (estas muestras se descartan)
- `chains=4`: 4 cadenas independientes para verificar convergencia (si convergen al mismo lugar, hay confianza)
- `target_accept=0.9`: tasa de aceptación deseada; valores más altos → pasos más pequeños pero más seguros
- `random_seed=42`: semilla aleatoria para reproducibilidad

> ⏱ **Nota:** Con 50,000 draws × 4 chains y 1000 observaciones, este proceso puede tomar varios minutos.

In [ ]:
# El muestreo se ejecuta dentro del contexto del modelo
# Esto es exactamente el mismo patrón del profesor:
#   with modelo: trace = pm.sample(...)

with modelo_multinomial:
    # pm.sample: ejecuta el algoritmo NUTS para obtener muestras de la distribución posterior
    # Cada muestra es un valor posible de (alpha, beta) según la información de los datos
    trace = pm.sample(
        draws=50000,           # muestras a guardar por cadena tras el calentamiento
        tune=5000,             # pasos de calentamiento (adaptación del paso NUTS, se descartan)
        chains=4,              # número de cadenas independientes
        target_accept=0.9,     # tasa de aceptación objetivo: mayor → más conservador
        random_seed=42,        # semilla: garantiza resultados reproducibles
        return_inferencedata=True  # devuelve objeto ArviZ InferenceData (formato moderno)
    )

print('\n✓ Muestreo MCMC completado.')

## 5. Diagnósticos MCMC

Antes de interpretar los coeficientes, **debemos verificar que las cadenas convergieron**. Un modelo que no converge da resultados incorrectos aunque no genere errores.

### Criterios de convergencia (del checklist del profesor):
1. **R-hat ≈ 1.0**: las 4 cadenas muestrearon la misma distribución
2. **Trazas estacionarias**: las cadenas mezclan bien sin tendencias
3. **Energy plot**: sin diferencias grandes entre transición y marginal de energía
4. **Sin divergencias**: no hay regiones del espacio posterior inaccesibles

### 5.1 Traceplot

In [ ]:
# az.plot_trace: genera dos paneles por parámetro
#   Izquierda: distribución posterior (KDE) de las 4 cadenas superpuestas
#   Derecha: trayectoria de muestreo a lo largo de las iteraciones
# Lo que buscamos en la traza derecha:
#   ✓ Las 4 cadenas se superponen (misma distribución)
#   ✓ No hay tendencias ni patrones (estacionariedad)
#   ✓ Las cadenas mezclan rápidamente (baja autocorrelación)

# Graficamos solo los interceptos para no saturar la figura
az.plot_trace(
    trace,
    var_names=['alpha'],  # interceptos: alpha[0]=Planner, alpha[1]=In-Between, alpha[2]=Last-Minute
    figsize=(12, 5)
)
plt.suptitle('Traceplot — Interceptos (alpha)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/05_traceplot_alpha.png', bbox_inches='tight')
plt.show()
print('Guardado: 05_traceplot_alpha.png')

In [ ]:
# Traceplot de los coeficientes beta
# beta tiene shape (n_pred, n_clases) = (7, 3) → 21 coeficientes
# compact=True agrupa las dimensiones en un solo panel por variable
az.plot_trace(
    trace,
    var_names=['beta'],
    compact=True,        # compactar dimensiones para no generar 21 paneles separados
    figsize=(14, 5)
)
plt.suptitle('Traceplot — Coeficientes (beta)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/06_traceplot_beta.png', bbox_inches='tight')
plt.show()
print('Guardado: 06_traceplot_beta.png')

### 5.2 R-hat y ESS

In [ ]:
# az.summary: tabla resumen completa de la distribución posterior
# Columnas clave:
#   mean:     media posterior (estimación puntual)
#   sd:       desviación estándar posterior (incertidumbre)
#   hdi_3%:   límite inferior del intervalo de máxima densidad al 94%
#   hdi_97%:  límite superior del intervalo de máxima densidad al 94%
#   r_hat:    factor de convergencia de Gelman-Rubin
#             r_hat ≈ 1.0 → convergencia perfecta
#             r_hat > 1.05 → NO converge, resultados no confiables
#   ess_bulk: tamaño efectivo de muestra (cuántas muestras independientes equivale)
#             ess_bulk > 400 → suficiente para estimaciones robustas
#   ess_tail: ESS en las colas (importante para los intervalos HDI)

resumen = az.summary(trace, var_names=['alpha', 'beta'], round_to=4)
print('Tabla de diagnósticos de convergencia:')
display(resumen[['mean', 'sd', 'hdi_3%', 'hdi_97%', 'r_hat', 'ess_bulk', 'ess_tail']])

# Evaluación automática de la convergencia
rhat_max = resumen['r_hat'].max()
ess_min  = resumen['ess_bulk'].min()

if rhat_max < 1.01:
    print(f'\n✓ Convergencia excelente: R-hat máximo = {rhat_max:.4f} (< 1.01)')
elif rhat_max < 1.05:
    print(f'\n⚠ Convergencia aceptable: R-hat máximo = {rhat_max:.4f} (< 1.05)')
else:
    print(f'\n✗ ADVERTENCIA: R-hat máximo = {rhat_max:.4f} — convergencia insuficiente')

print(f'ESS bulk mínimo: {ess_min:.0f} muestras efectivas')

### 5.3 Energy Plot y divergencias

In [ ]:
# az.plot_energy: compara la distribución de energía de las transiciones (barras)
# con la distribución marginal de energía (línea)
# Si ambas distribuciones son similares → el sampler explora bien el espacio posterior
# Si difieren mucho → hay regiones inaccesibles (patología en el modelo)
az.plot_energy(trace, figsize=(8, 4))
plt.title('Energy Plot — Exploración del Espacio Posterior')
plt.savefig(f'{IMG_DIR}/07_energy_plot.png', bbox_inches='tight')
plt.show()
print('Guardado: 07_energy_plot.png')

# Contar divergencias: ocurren cuando NUTS no puede seguir la curvatura del posterior
# Causas comunes: modelo mal especificado, priors demasiado estrechos, colinealidad extrema
# Muchas divergencias (> 1% del total) indican resultados no confiables
n_div = int(trace.sample_stats.diverging.sum())
total_samples = 50000 * 4  # draws × chains
pct_div = n_div / total_samples * 100

if n_div == 0:
    print(f'\n✓ Sin divergencias (0 de {total_samples:,} muestras)')
elif pct_div < 1:
    print(f'\n⚠ {n_div} divergencias ({pct_div:.2f}% del total) — revisar modelo')
else:
    print(f'\n✗ {n_div} divergencias ({pct_div:.2f}% del total) — modelo problemático')

## 6. Resultados Posteriores

### 6.1 Distribuciones posteriores de los interceptos

Los interceptos `alpha[c]` representan el log-odds de pertenecer a la clase $c$ cuando **todos los predictores son cero** (es decir, cuando el cliente es "promedio" en todas las variables estandarizadas y las binarias son 0).

In [ ]:
# az.plot_posterior: distribución posterior con media, HDI y densidad
# hdi_prob=0.95: muestra el Highest Density Interval del 95%
#   El HDI es el intervalo más estrecho que contiene el 95% de la masa posterior
#   (diferente al IC frecuentista: aquí interpretamos como
#    "hay 95% de probabilidad de que el parámetro esté en este rango")
# ref_val=0: línea vertical en cero para ver si el parámetro es positivo/negativo/neutro

az.plot_posterior(
    trace,
    var_names=['alpha'],
    hdi_prob=0.95,
    ref_val=0,           # línea en cero: si el HDI no cruza cero, el efecto es robusto
    figsize=(14, 4),
    textsize=10
)
# Renombramos los títulos de los subplots para mayor claridad
for ax, label in zip(plt.gcf().axes, ['alpha[0] — Planner', 'alpha[1] — In-Between', 'alpha[2] — Last-Minute']):
    ax.set_title(label)

plt.suptitle('Distribuciones Posteriores — Interceptos por Clase (HDI 95%)', y=1.05, fontsize=13)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/08_posterior_interceptos.png', bbox_inches='tight')
plt.show()
print('Guardado: 08_posterior_interceptos.png')

### 6.2 Forest Plot de coeficientes por clase

El forest plot muestra simultáneamente los intervalos HDI de todos los coeficientes. Es la visualización clave para comparar la magnitud e incertidumbre de cada predictor.

**Interpretación:** Si el HDI de `beta[j, c]` **no cruza el cero**, hay evidencia sólida de que el predictor $j$ discrimina a los clientes hacia/contra la clase $c$.

In [ ]:
# az.plot_forest: intervalos HDI de todos los coeficientes en un solo gráfico
# combined=True: combina las 4 cadenas en una sola estimación
# hdi_prob=0.95: intervalos al 95%
# r_hat=True: muestra el valor de R-hat junto a cada parámetro (diagnóstico rápido)

az.plot_forest(
    trace,
    var_names=['beta'],
    combined=True,
    hdi_prob=0.95,
    r_hat=True,           # muestra R-hat al lado de cada parámetro
    figsize=(10, 12)
)
# Línea vertical roja en cero: referencia para identificar efectos positivos/negativos
plt.axvline(0, color='red', linestyle='--', linewidth=1.2, label='Efecto nulo (β=0)')
plt.title('Forest Plot — Coeficientes beta (HDI 95%)\nbeta[predictor, clase]: 0=Planner, 1=In-Between, 2=Last-Minute', fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/09_forest_plot_beta.png', bbox_inches='tight')
plt.show()
print('Guardado: 09_forest_plot_beta.png')

In [ ]:
# Forest plot de los interceptos junto a la línea de cero
az.plot_forest(
    trace,
    var_names=['alpha'],
    combined=True,
    hdi_prob=0.95,
    r_hat=True,
    figsize=(8, 3)
)
plt.axvline(0, color='red', linestyle='--', linewidth=1.2)
plt.title('Forest Plot — Interceptos alpha (HDI 95%)', fontsize=11)
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/10_forest_plot_alpha.png', bbox_inches='tight')
plt.show()
print('Guardado: 10_forest_plot_alpha.png')

### 6.3 Tabla resumen de coeficientes con nombres de predictores

In [ ]:
# Construimos una tabla legible con los nombres de los predictores
# para facilitar la interpretación del forest plot

nombres_pred = X_df.columns.tolist()  # nombres de los 7 predictores
nombres_clases = list(mapa_clases.keys())  # ['Planner', 'In-Between', 'Last-Minute']

# Extraer medias posteriores y HDI de los coeficientes beta
beta_posterior = trace.posterior['beta']  # shape: (chains, draws, n_pred, n_clases)
beta_mean = beta_posterior.mean(dim=['chain', 'draw']).values  # (n_pred, n_clases)

# Construir DataFrame para visualización
tabla_coef = pd.DataFrame(
    beta_mean,
    index=nombres_pred,
    columns=[f'β — {c}' for c in nombres_clases]
)

print('Media posterior de coeficientes beta por clase:')
print('(valores positivos = mayor probabilidad de esa clase, negativos = menor probabilidad)')
display(tabla_coef.round(3))

# Identificar el predictor con mayor efecto absoluto promedio
efecto_abs = tabla_coef.abs().mean(axis=1)
pred_mayor = efecto_abs.idxmax()
print(f'\nPredictor con mayor efecto medio: {pred_mayor} (efecto abs. promedio = {efecto_abs[pred_mayor]:.3f})')

## 7. Posterior Predictive Check (PPC)

El PPC verifica si el modelo puede regenerar datos similares a los observados. Siguiendo el estilo del profesor (como en `RegresionLineal.ipynb` con `pm.sample_posterior_predictive`), generamos predicciones sobre el conjunto de prueba.

In [ ]:
# Para predecir sobre el conjunto de PRUEBA necesitamos un nuevo modelo
# que use X_test en lugar de X_train como datos observados
# Esto es el patrón estándar de PyMC para predicción fuera de muestra

with pm.Model() as modelo_pred:
    # Mismos priors que el modelo de entrenamiento
    alpha_pred = pm.Normal('alpha', mu=0, sigma=5, shape=n_clases)
    beta_pred  = pm.Normal('beta', mu=0, sigma=5, shape=(n_pred, n_clases))

    # Modelo lineal ahora con X_TEST (datos no vistos durante el entrenamiento)
    mu_pred = pm.math.dot(X_test, beta_pred) + alpha_pred

    # Softmax sobre el conjunto de prueba
    p_pred = pm.math.softmax(mu_pred, axis=-1)

    # Verosimilitud sobre y_test (para PPC)
    y_obs_pred = pm.Categorical('y_obs', p=p_pred, observed=y_test)

    # pm.set_data no aplica aquí, en su lugar usamos pm.sample_posterior_predictive
    # con el trace del modelo de entrenamiento
    # Esto "congela" los parámetros en sus valores posteriores y genera predicciones
    ppc = pm.sample_posterior_predictive(
        trace,             # trace del modelo entrenado: contiene las muestras de alpha y beta
        random_seed=42     # reproducibilidad
    )

print('✓ Posterior Predictive Check completado.')

### 7.1 Probabilidades predichas por clase

In [ ]:
# Las predicciones del PPC son muestras de la distribución predictiva posterior
# ppc.posterior_predictive['y_obs'] tiene forma (chains, draws, n_test)
# Cada valor es una clase predicha (0, 1 o 2) muestreada del posterior

# Para obtener probabilidades por clase, calculamos la frecuencia con que
# el modelo predice cada clase para cada observación del test set

# y_pred_samples tiene forma (chains*draws, n_test)
y_pred_samples = ppc.posterior_predictive['y_obs'].values.reshape(-1, len(y_test))

# Calcular probabilidad de cada clase para cada observación
# prob[i, c] = fracción de muestras donde el modelo predijo clase c para la observación i
prob_pred = np.stack([
    (y_pred_samples == c).mean(axis=0)  # promedio sobre todas las muestras MCMC
    for c in range(n_clases)
], axis=1)  # resultado: (n_test, 3)

# La clase predicha es la que tiene mayor probabilidad posterior
y_pred_clase = prob_pred.argmax(axis=1)  # argmax sobre las 3 clases

# DataFrame con probabilidades predichas
df_pred = pd.DataFrame(
    prob_pred,
    columns=[f'P({c})' for c in nombres_clases]
)
df_pred['Clase_real']    = [nombres_clases[c] for c in y_test]
df_pred['Clase_pred']    = [nombres_clases[c] for c in y_pred_clase]
df_pred['Correcta']      = df_pred['Clase_real'] == df_pred['Clase_pred']

print('Muestra de probabilidades predichas (primeras 10 observaciones del test):')
display(df_pred.head(10).round(3))
print(f'\nAccuracy: {df_pred["Correcta"].mean():.4f} ({df_pred["Correcta"].mean()*100:.1f}%)')

### 7.2 Matriz de confusión

La matriz de confusión muestra cuántas predicciones fueron correctas (diagonal) y cuáles se confundieron entre clases (fuera de la diagonal). Es la misma métrica que el profesor usa en `Modelos_Multivariados_2.ipynb` para el clasificador spam/no-spam.

In [ ]:
# confusion_matrix: matriz n_clases × n_clases
# conf[i, j] = número de observaciones reales de clase i predichas como clase j
# La diagonal principal (conf[i, i]) = predicciones correctas
conf = confusion_matrix(y_test, y_pred_clase)

# Visualizar con heatmap anotado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap de conteos absolutos
sns.heatmap(
    conf,
    annot=True,           # mostrar números en cada celda
    fmt='d',              # formato entero
    cmap='Blues',         # escala de azules: más oscuro = más observaciones
    xticklabels=nombres_clases,
    yticklabels=nombres_clases,
    ax=axes[0]
)
axes[0].set_title('Matriz de Confusión (conteos)')
axes[0].set_xlabel('Clase predicha')
axes[0].set_ylabel('Clase real')

# Heatmap de proporciones (normalizado por fila = recall por clase)
conf_norm = conf.astype(float) / conf.sum(axis=1, keepdims=True)  # normalizar por fila
sns.heatmap(
    conf_norm,
    annot=True,
    fmt='.2%',            # formato porcentaje
    cmap='Blues',
    xticklabels=nombres_clases,
    yticklabels=nombres_clases,
    ax=axes[1]
)
axes[1].set_title('Matriz de Confusión (% por fila = Recall)')
axes[1].set_xlabel('Clase predicha')
axes[1].set_ylabel('Clase real')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/11_matriz_confusion.png', bbox_inches='tight')
plt.show()
print('Guardado: 11_matriz_confusion.png')

# Reporte de clasificación completo (igual que el profe en spam)
# Incluye: precision, recall, F1-score y soporte por clase
print('\nReporte de Clasificación:')
print(classification_report(y_test, y_pred_clase, target_names=nombres_clases))

### 7.3 Visualización de probabilidades predichas por clase real

In [ ]:
# Scatter de probabilidades predichas: cada punto es un cliente del test set
# Eje X: probabilidad predicha de ser Planner
# Eje Y: probabilidad predicha de ser Last-Minute
# Color: clase real del cliente
# Si el modelo discrimina bien, las clases deberían separarse en el espacio de probabilidades

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colores_clase = {'Planner': '#2196F3', 'In-Between': '#FF9800', 'Last-Minute': '#4CAF50'}

# Scatter: P(Planner) vs P(Last-Minute)
for clase in nombres_clases:
    mask = df_pred['Clase_real'] == clase
    axes[0].scatter(
        df_pred.loc[mask, 'P(Planner)'],
        df_pred.loc[mask, 'P(Last-Minute)'],
        alpha=0.5, label=clase,
        color=colores_clase[clase],
        edgecolors='white', linewidths=0.3, s=40
    )
axes[0].set_xlabel('P(Planner)')
axes[0].set_ylabel('P(Last-Minute)')
axes[0].set_title('Probabilidades predichas: Planner vs Last-Minute')
axes[0].legend()

# Stacked bar: probabilidad promedio predicha por clase real
prob_por_clase_real = df_pred.groupby('Clase_real')[['P(Planner)', 'P(In-Between)', 'P(Last-Minute)']].mean()
prob_por_clase_real.plot(
    kind='bar', stacked=True,
    color=['#2196F3', '#FF9800', '#4CAF50'],
    ax=axes[1], edgecolor='white'
)
axes[1].set_title('Probabilidad media predicha por clase real')
axes[1].set_xlabel('Clase real')
axes[1].set_ylabel('Probabilidad promedio')
axes[1].legend(title='Clase predicha')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/12_probabilidades_predichas.png', bbox_inches='tight')
plt.show()
print('Guardado: 12_probabilidades_predichas.png')

## 8. Perfiles de Compradores

Respondemos las preguntas orientadoras del taller basándonos en los coeficientes posteriores.

In [ ]:
# Heatmap de coeficientes medios: muestra el efecto de cada predictor en cada clase
# Colores cálidos (rojo) = coeficiente positivo → mayor probabilidad de esa clase
# Colores fríos (azul) = coeficiente negativo → menor probabilidad de esa clase

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    tabla_coef.T,           # transpuesta: clases en filas, predictores en columnas
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',          # divergente: rojo=positivo, azul=negativo
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Media Posterior de Coeficientes β por Clase y Predictor', fontsize=12)
ax.set_xlabel('Predictor')
ax.set_ylabel('Clase')
plt.tight_layout()
plt.savefig(f'{IMG_DIR}/13_heatmap_coeficientes.png', bbox_inches='tight')
plt.show()
print('Guardado: 13_heatmap_coeficientes.png')

In [ ]:
# ---- RESUMEN INTERPRETATIVO AUTOMÁTICO ----

def get_beta_stats(pred_idx, clase_idx):
    """Retorna (media, hdi_low, hdi_high) del coeficiente beta[pred, clase]."""
    # Extraemos todas las muestras de ese coeficiente específico
    # trace.posterior['beta'] tiene dims (chain, draw, predictor, clase)
    vals = trace.posterior['beta'].values[:, :, pred_idx, clase_idx].flatten()
    mean_val = float(vals.mean())
    hdi = az.hdi(vals, hdi_prob=0.95)  # array 1D con [low, high]
    return mean_val, float(hdi[0]), float(hdi[1])

def cruza_cero(low, high):
    return low < 0 < high

# Índice 1 = Days_Before_Concierto_z (el más discriminante esperado)
idx_dias = nombres_pred.index('Days_Before_Concierto_z')

print('=' * 72)
print('PERFILES DE COMPRADORES — REGRESIÓN MULTINOMIAL BAYESIANA')
print('Movistar Arena — Concierto Miguel Amezquita')
print('=' * 72)

for c_idx, c_nombre in enumerate(nombres_clases):
    print(f'\n{"─"*70}')
    print(f'CLASE {c_idx}: {c_nombre.upper()}')
    print(f'{"─"*70}')
    for p_idx, p_nombre in enumerate(nombres_pred):
        m, lo, hi = get_beta_stats(p_idx, c_idx)
        signo = '+' if m > 0 else '-'
        conclusión = '✓ Efecto robusto' if not cruza_cero(lo, hi) else '⚠ Efecto incierto'
        print(f'  {p_nombre:<35s}: β={m:+.3f}  [HDI95%: {lo:.3f}, {hi:.3f}]  {conclusión}')

print(f'\n{"═"*72}')
print('RESPUESTAS A LAS PREGUNTAS ORIENTADORAS DEL TALLER')
print(f'{"═"*72}')

# Q1: ¿Qué caracteriza al Planner?
m_dias_plan, lo_dias_plan, hi_dias_plan = get_beta_stats(idx_dias, 0)
print(f"""
1. ¿Qué características asocian con compradores que planean con anticipación?
   → Days_Before_Concierto: β={m_dias_plan:+.3f} para Planner
     {'Positivo y robusto: mayor anticipación → mayor P(Planner)' if not cruza_cero(lo_dias_plan, hi_dias_plan) and m_dias_plan > 0 else 'Efecto incierto: el HDI cruza el cero'}
""")

# Q2: ¿Qué señales identifican al Last-Minute?
m_dias_lm, lo_dias_lm, hi_dias_lm = get_beta_stats(idx_dias, 2)
print(f"""2. ¿Qué señales identifican a quienes compran a última hora?
   → Days_Before_Concierto: β={m_dias_lm:+.3f} para Last-Minute
     {'Negativo y robusto: mayor anticipación → MENOR P(Last-Minute)' if not cruza_cero(lo_dias_lm, hi_dias_lm) and m_dias_lm < 0 else 'Efecto incierto sobre Last-Minute'}
""")

# Q3: Predictor más discriminante
print(f"""3. ¿Cuál es el predictor con mayor poder discriminante?
   → {pred_mayor}: efecto absoluto promedio = {efecto_abs[pred_mayor]:.3f}
     Este predictor es el que más diferencia el perfil de compra entre los tres grupos.
""")

# Q4: Segmentación para diseño de campaña
accuracy = df_pred['Correcta'].mean()
print(f"""4. ¿Cómo apoya esta segmentación el diseño de campañas y logística?
   El modelo alcanza una accuracy del {accuracy*100:.1f}% en el conjunto de prueba.
   - Planners: estrategias de fidelización temprana, preventa con descuento,
     comunicación via Fan Mailing List.
   - Last-Minute: alertas de últimas boletas, precios dinámicos, urgencia.
   - In-Between: campañas de recordatorio en la semana previa al evento.
""")

print('=' * 72)

## Resumen de figuras generadas

In [ ]:
import glob
figuras = sorted(glob.glob(f'{IMG_DIR}/*.png'))
print(f'Figuras generadas en {IMG_DIR}/ ({len(figuras)} archivos):')
for f in figuras:
    print(f'  {os.path.basename(f)}')